## Data Extraction

In [2]:
import io
import sys
import os
import json
import unicodedata
import subprocess
import boto3
import shutil
from docx import Document
from spire.doc import Document as SpireDocument
from spire.doc import FileFormat
import win32com.client
from sentence_transformers import SentenceTransformer
import pdfplumber
from lxml import etree
from pinecone import Pinecone
from tqdm import tqdm
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
fuckass_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\data\raw_data\'

In [5]:
def convert_doc_to_docx(RAW_DATA):
    for filename in os.listdir(RAW_DATA):
        if filename.lower().endswith('.doc') and not filename.lower().endswith('.docx'):
            input_path = os.path.join(RAW_DATA, filename)
            output_path = os.path.splitext(input_path)[0] + ".docx"

            if not os.path.exists(input_path):
                print(f"File not found: {input_path}")
                continue

            print(f"Converting {filename} to .docx...")

            try:
                doc = SpireDocument()
                doc.LoadFromFile(input_path)
                doc.SaveToFile(output_path, FileFormat.Docx2016)
                doc.Close()
                print(f"Converted {filename} to .docx successfully.")

            except Exception as e:
                print(f"Failed to convert {filename} with Spire.Doc: {e}")

convert_doc_to_docx(fuckass_file)

NotADirectoryError: [WinError 267] The directory name is invalid: 'C:\\Users\\txcjs\\OneDrive\\Documents\\Homework\\Yr 3.1\\ICP\\Beep_Boop\\data\\raw_data\\2_Verztec Webmail and Autoresponder.doc'

In [ ]:
import os
from docx import Document
import win32com.client
from sentence_transformers import SentenceTransformer
from pprint import pprint
import PyPDF2
from pptx import Presentation
import openpyxl
import pdfplumber
from lxml import etree
from bs4 import BeautifulSoup
from pinecone import Pinecone
from tqdm import tqdm
from langchain.text_splitter import RecursiveCharacterTextSplitter
import unicodedata

In [2]:
FILE_PATH = r'../data/raw_data'

for file in os.listdir(FILE_PATH):
    print(file)

11A_Basic Meeting Etiquette for Professionals.pdf
18_Verztec_Pantry Rules_060715.pdf
26. Import Supplier E-Invoice from xtranet to ABSS Purchase Module.docx
26_Policy on Office Laptop and Computer_050820.pdf
28_Verztec Digital Meeting Etiquette.pdf
2_Verztec Webmail and Autoresponder.doc
3_Offboarding Process on Clean Desk Policy_150125.pdf
4_Verztec Ownership Policy_280420.pdf
8_Basic Telephone Skills.pdf
Project Sign off and Change Acknowledgement Form_250227 .docx
SOP checklist transcription projects_270225.docx


In [25]:
doc_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\test_data\SampleDOCFile_200kb.doc"

In [ ]:
pdf_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\data\raw_data\3_Offboarding Process on Clean Desk Policy_150125.pdf"

In [48]:
docx_file_path = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\data\raw_data\26. Import Supplier E-Invoice from xtranet to ABSS Purchase Module.docx"

In [49]:
def extract_text_from_docx(file_path):
    """Extracts .docx files, ignores images"""
    doc = Document(file_path)
    full_text = []

    # Get the raw XML of the Word doc so we can manually look at paragraphs, tables, etc.
    doc_xml = doc.element
    namespaces = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}

    for element in doc_xml.body:
        # Extract text
        if element.tag == etree.QName(namespaces['w'], 'p'):
            # Detect if it's a list item
            num_pr = element.find('.//w:numPr', namespaces)
            is_list = num_pr is not None

            # Extract hyperlink stuff
            hyperlink = element.find('.//w:hyperlink', namespaces)
            if hyperlink is not None:

                # Hyperlink text
                texts = hyperlink.findall('.//w:t', namespaces)
                link_text = ''.join(t.text for t in texts if t.text)

                # Get the hyperlink target
                r_id = hyperlink.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
                if r_id:
                    rels = doc.part.rels
                    url = rels[r_id]._target if r_id in rels else ''
                    plain_text = f"[{link_text}]({url})"
                else:
                    # No url found
                    plain_text = link_text
            else:
                # Plain text
                texts = element.findall('.//w:t', namespaces)
                plain_text = ''.join(t.text for t in texts if t.text)

            plain_text = plain_text.strip()
            if plain_text:
                if is_list:
                    plain_text = f"• {plain_text}"
                full_text.append(plain_text)

        # Extract tables
        elif element.tag == etree.QName(namespaces['w'], 'tbl'):
            for row in element.findall('.//w:tr', namespaces):
                cells = row.findall('.//w:tc', namespaces)
                row_cells = []
                for cell in cells:
                    texts = cell.findall('.//w:t', namespaces)
                    cell_text = ''.join(t.text for t in texts if t.text).strip()
                    row_cells.append(cell_text)
                formatted_row = '| ' + ' | '.join(row_cells) + ' |'
                full_text.append(formatted_row)

    return '\n'.join(full_text)

# Usage
extracted_text = extract_text_from_docx(docx_file_path)
print(extracted_text)

Date: 13/07/2020
How to import supplier e-invoice data file from xtranet to ABSS purchase transaction module
• As per normal procedure, go to “View Invoice” menu to Print a copy of the invoice with approve the e-invoice thereafter. The hard copy of invoice will be filed to unpaid creditor file after we have checked and confirmed that the e-invoice data has been imported to ABSS purchase transaction module successfully.
• At Invoice Module, there would be 2 New Sub-Menus created under this module now.
• Generate New ABSS File – This is use to generate invoice data file into ABSS file in csv format (excel format first)
• Exported ABSS File – After the invoice data file is being imported to ABSS, the data file will be moved from Generate New ABSS File menu to here as past record
• Click on “Generate New ABSS File” and you will see there is a list of e-invoice data file show at below
• Check on the box near to S/N as to select all invoices for importing
• Click “Generate All to ABSS” box a

In [24]:
def extract_text_from_doc(file_path):
    """Extracts .doc files, ignores images"""

    # Have to open the file in background because its old -_-
    word = win32com.client.Dispatch("Word.Application")
    word.Visible = False
    doc = word.Documents.Open(file_path)

    full_text = []
    content = doc.Content
    start = content.Start
    end = content.End
    bullets = {'•', '‣', '·', '‧', '–', '-', '*', '', '●', '■', '♦', '\uf0b7', 'o'}

    while start < end:
        current_range = doc.Range(start, start + 1)
        # Extract tables
        if current_range.Tables.Count > 0:
            table = current_range.Tables(1)
            table_text = []
            for row in table.Rows:
                row_text = []
                for cell in row.Cells:
                    cell_text = cell.Range.Text.strip().replace('\r', '').replace('\x07', '')
                    row_text.append(cell_text)
                table_text.append(' | '.join(row_text))
            full_text.append('\n'.join(table_text))
            start = table.Range.End

        # Extract paragraphs
        elif current_range.Paragraphs.Count > 0:
            para_range = current_range.Paragraphs(1).Range
            para_text = para_range.Text.strip().replace('\r', '').replace('\x07', '')
            para_text = unicodedata.normalize("NFKC", para_text) # Normalize to raw text

            if para_text:
                list_format = para_range.ListFormat
                indent = ''

                '''This chunk of code is to deal with microsoft word lists'''
                if list_format.ListType != 0:
                    # Get list indent level and marker
                    level = max(list_format.ListLevelNumber, 1)
                    indent = '    ' * (level - 1)
                    marker = list_format.ListString.strip()

                    # Some markers are invisible or from Wingdings/Symbol font (like '\uf0b7')
                    # These don't render well, so we substitute a standard bullet
                    if not marker.isprintable() or ord(marker[0]) >= 0xF000:
                        marker = '•'
                    para_text = f"{indent}{marker} {para_text}"

                else:
                    stripped = para_text.lstrip()
                    # Now check if the first character is a bullet point
                    if stripped and stripped[0] in bullets:
                        para_text = f"• {stripped[1:].lstrip()}"

                full_text.append(para_text)

            start = para_range.End
        else:
            start = current_range.End  # Fallback to avoid infinite loop

    doc.Close(False)
    word.Quit()
    return '\n\n'.join(full_text)

extracted_text = extract_text_from_doc(doc_file_path)
print(extracted_text)

Verztec Webmail

When you are not in the office, you may access Verztec Webmail at this URL:

http://webmail.verztec.com

Userid: your email address

Password: your password

This is useful when you need to check emails from home/ outside to reply to your clients/ suppliers.

Please do check emails regularly when you are accessing Internet at home.

Do take note of there will be email relay problems when you send out. As always, please put yourself as one of the recipients to ensure the email REALLY went out.

Verztec SMTP mail Server for use in office MS Outlook:

Incoming and outgoing: mail.verztec.com

Username: Your email address.

• * Note: If you download mails to your home PC using MS Outlook, you will not be able to download the same emails using your office PC - MS Outlook, so please use Webmail from home to keep a copy of the mails. Note that once you’ve downloaded your emails to your Outlook, it will not keep a copy in the Webmail / Server anymore.

If you have problems send

In [ ]:
def extract_text_from_pdf(file_path):
    """Extracts PDF, ignores images"""
    output = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            # Extract plain text
            plain_text = page.extract_text()
            if plain_text:
                output.append(plain_text.strip())

            # Extract tables
            tables = page.extract_tables()
            for table in tables:
                formatted_table = []
                for row in table:
                    row_text = " | ".join(cell.strip() if cell else "" for cell in row)
                    formatted_table.append(f"| {row_text} |")
                output.append("\n".join(formatted_table))

    return "\n\n".join(output)
extracted_text = extract_text_from_pdf(pdf_file_path)
pprint(extracted_text)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


('Date: 15th JAN 2025\n'
 'Offboarding Clean Desk and Digital Handover Policy\n'
 'This policy outlines the steps employees must take to maintain a clean and '
 'organized workspace and ensure\n'
 'all necessary digital files are properly backed up and handed over before '
 'their final day of work.\n'
 'Clean Desk Policy\n'
 'Employees must ensure their workspace is clear of personal and unnecessary '
 'items by the end of their final\n'
 'working day. This includes:\n'
 '1. Removal of Personal Belongings:\n'
 'o Take home all personal items such as photos, decorations, and personal '
 'stationery etc.\n'
 'o Check and empty all drawers, cabinets, and other storage areas for '
 'personal belongings.\n'
 'o Bring back/ throw away any food or drinks you stored in the office '
 'refrigerator.\n'
 '2. Organizing Work Materials:\n'
 'o Sort through physical documents. Shred or dispose of sensitive documents '
 'no longer\n'
 'needed. Remove all name cards and old files outside the bin near

## Data Chunking

In [50]:
def split_and_print_chunks(text, chunk_size=1000, chunk_overlap=200):
    """
    Splits the input text into chunks and prints each chunk with its index.

    Parameters:
        text (str): The text to be split.
        chunk_size (int): The maximum size of each chunk.
        chunk_overlap (int): The number of overlapping characters between chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_text(extracted_text)


    print(f"Total chunks: {len(chunks)}\n")

    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i + 1} ---\n{chunk}\n{'-' * 60}")

    return chunks
chunks = split_and_print_chunks(extracted_text, chunk_size=1000, chunk_overlap=200)

Total chunks: 5


--- Chunk 1 ---
Date: 13/07/2020
How to import supplier e-invoice data file from xtranet to ABSS purchase transaction module
• As per normal procedure, go to “View Invoice” menu to Print a copy of the invoice with approve the e-invoice thereafter. The hard copy of invoice will be filed to unpaid creditor file after we have checked and confirmed that the e-invoice data has been imported to ABSS purchase transaction module successfully.
• At Invoice Module, there would be 2 New Sub-Menus created under this module now.
• Generate New ABSS File – This is use to generate invoice data file into ABSS file in csv format (excel format first)
• Exported ABSS File – After the invoice data file is being imported to ABSS, the data file will be moved from Generate New ABSS File menu to here as past record
• Click on “Generate New ABSS File” and you will see there is a list of e-invoice data file show at below
• Check on the box near to S/N as to select all invoices for importing
--

In [ ]:
def embedding_chunks(file_path, chunks, model_name = 'BAAI/bge-large-en'):
    """
    Encodes text chunks into embeddings and structures them with metadata.

    Parameters:
        file_path (str): Path to the source file (used for metadata).
        chunks (list): List of text chunks.
        model_name (str): Name of the sentence transformer model to use.

    Returns:
        list: A list of dictionaries containing embeddings and metadata.
    """
    filename = os.path.basename(file_path)
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks)

    vector_data = [
        {
            "id": f"{filename}_chunk{i}",
            "values": embeddings[i].tolist(),
            "metadata": {
                "source": filename,
                "text": chunks[i],
                "chunk": i,
                "chunk_size": len(chunks[i])
            }
        }
        for i in range(len(chunks))
    ]

    print(f"\nEmbedded {len(vector_data)} chunks.\n")
    print("--- Sample chunk ---")
    print(vector_data[0]['metadata']['text'])
    print(f"\n--- Embedding length: {len(vector_data[0]['values'])} ---")

    return vector_data

vector_data = embedding_chunks(pdf_file_path, chunks, model_name = 'BAAI/bge-large-en')


Embedded 5 chunks.

--- Sample chunk ---
Date: 15th JAN 2025
Offboarding Clean Desk and Digital Handover Policy
This policy outlines the steps employees must take to maintain a clean and organized workspace and ensure
all necessary digital files are properly backed up and handed over before their final day of work.
Clean Desk Policy
Employees must ensure their workspace is clear of personal and unnecessary items by the end of their final
working day. This includes:
1. Removal of Personal Belongings:
o Take home all personal items such as photos, decorations, and personal stationery etc.
o Check and empty all drawers, cabinets, and other storage areas for personal belongings.
o Bring back/ throw away any food or drinks you stored in the office refrigerator.
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.

--- Embedding length: 1024 ---


In [39]:
file_path = pdf_file_path
filename = os.path.basename(file_path)
filename

'3_Offboarding Process on Clean Desk Policy_150125.pdf'

In [ ]:
PINECONE_API_KEY = "pcsk_4Zrgdk_JE48SUN5TDzkNnqYWdbszMCwwkJpQpQLq5MxQDw4a7vJGyiWEMeEJMWhv9CWADB"
PINECONE_ENV = "us-east-1"
INDEX_NAME = "internal-docs"


def upsert_to_pinecone(vector_data, index, batch_size=100, namespace=None):
    """ Uploads a list of vectors to a Pinecone index """
    print(f"\nStarting upsert to Pinecone index '{INDEX_NAME}' (batch size: {batch_size})...")

    for start in tqdm(range(0, len(vector_data), batch_size), desc="Upserting to Pinecone"):
        batch = vector_data[start:start + batch_size]
        index.upsert(vectors=batch, namespace=namespace)

    print(f"\nUploaded {len(vector_data)} vectors to Pinecone index '{INDEX_NAME}'.")
pc = Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)
index = pc.Index(INDEX_NAME)
upsert_to_pinecone(vector_data, index, batch_size = 100, namespace = None)


Starting upsert to Pinecone index 'internal-docs' (batch size: 100)...


Upserting to Pinecone: 100%|██████████| 1/1 [00:02<00:00,  2.09s/it]


Uploaded 5 vectors to Pinecone index 'internal-docs'.


In [42]:
response = index.fetch(ids=["3_Offboarding Process on Clean Desk Policy_150125.pdf_chunk0"])
pprint(response)

FetchResponse(namespace='',
              vectors={'3_Offboarding Process on Clean Desk Policy_150125.pdf_chunk0': Vector(id='3_Offboarding '
                                                                                                 'Process '
                                                                                                 'on '
                                                                                                 'Clean '
                                                                                                 'Desk '
                                                                                                 'Policy_150125.pdf_chunk0',
                                                                                              values=[-0.0215702038,
                                                                                                      -0.0273689236,
                                                                             

In [44]:
query = "What should I do to maintain a clean and organized workspace?"
model = SentenceTransformer('BAAI/bge-large-en')
query_vector = model.encode([query])[0].tolist()

result = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

for match in result['matches']:
    print(f"\n--- ID: {match['id']} | Score: {match['score']} ---")
    print(match['metadata']['text'])



--- ID: 3_Offboarding Process on Clean Desk Policy_150125.pdf_chunk1 | Score: 0.849586725 ---
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.
o Return all company property (e.g., laptop, keyboard, employment card, medical card, office
keys etc) to the designated person or department.
o Please ensure that sensitive documents are not left on display, which can cause data theft
and leaks.
3. Desk Equipment:
o Do a clean up and leave the desk and its equipment (e.g., monitors, keyboards etc) clean
and functional.
Digital File Handover
Employees must ensure that all relevant digital files are properly backed up and handed over. The following
steps should be completed:
1. Backup Necessary Files:
o Transfer all work-related files to the designated shared drive or cloud storage.
o Ensure files are organized logically in folders for easy access.
2. 

### Chunks + Embed

In [ ]:
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# === Config ===
embed_file = r'C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Beep_Boop\data\clean_data\formatted_text.txt'
filename = os.path.basename(embed_file)
model = SentenceTransformer('BAAI/bge-large-en')  # 1024-dimensional embeddings

# === Load the file ===
with open(embed_file, 'r', encoding='utf-8') as f:
    raw_text = f.read()

# === Chunk using LangChain ===
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_text(raw_text)

# === Embed chunks ===
embeddings = model.encode(chunks)

# === Combine into vector list (ready for Pinecone or saving) ===
vector_data = [
    {
        "id": f"{filename}_chunk{i}",
        "values": embeddings[i].tolist(),
        "metadata": {
            "source": filename,
            "text": chunks[i],
            "chunk": i,
            "chunk_size": len(chunks[i])
        }
    }
    for i in range(len(chunks))
]

# === Optional: Show one entry ===
print(f"\nEmbedded {len(vector_data)} chunks.\n")
print("--- Sample chunk ---")
print(vector_data[0]['metadata']['text'])
print(f"\n--- Embedding length: {len(vector_data[0]['values'])} ---")



✅ Embedded 4 chunks.

--- Sample chunk ---
Date: 15th JAN 2025
Offboarding Clean Desk and Digital Handover Policy
This policy outlines the steps employees must take to maintain a clean and organized workspace and ensure
all necessary digital files are properly backed up and handed over before their final day of work.
Clean Desk Policy
Employees must ensure their workspace is clear of personal and unnecessary items by the end of their final
working day. This includes:
1. Removal of Personal Belongings:
o Take home all personal items such as photos, decorations, and personal stationery etc.
o Check and empty all drawers, cabinets, and other storage areas for personal belongings.
o Bring back/ throw away any food or drinks you stored in the office refrigerator.
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.

--- Embedding length: 1024 ---


In [27]:
vector_data[1]['id']

'formatted_text.txt_1'

In [22]:
for i, item in enumerate(vector_data):
    print(f"\n--- Chunk {i} ---")
    print(f"ID: {item['id']}")
    print(f"Chunk #: {item['metadata']['chunk']}")
    print(f"Source: {item['metadata']['source']}")
    print(f"Text preview: {item['metadata']['text'][:50]}...")  # First 200 chars
    print(f"Embedding length: {len(item['values'])}")
    print(f"Embedding sample: {item['values'][:5]}")  # First 5 values
    print("-" * 60)



--- Chunk 0 ---
ID: formatted_text.txt_0
Chunk #: 0
Source: formatted_text.txt
Text preview: Date: 15th JAN 2025
Offboarding Clean Desk and Dig...
Embedding length: 1024
Embedding sample: [-0.021570194512605667, -0.027368921786546707, -0.011659665033221245, 0.04162769019603729, 0.002682835329324007]
------------------------------------------------------------

--- Chunk 1 ---
ID: formatted_text.txt_1
Chunk #: 1
Source: formatted_text.txt
Text preview: 2. Organizing Work Materials:
o Sort through physi...
Embedding length: 1024
Embedding sample: [0.0013925275998190045, -0.007303130347281694, 0.00020290228712838143, 0.02757474221289158, -0.007039081305265427]
------------------------------------------------------------

--- Chunk 2 ---
ID: formatted_text.txt_2
Chunk #: 2
Source: formatted_text.txt
Text preview: o Transfer all work-related files to the designate...
Embedding length: 1024
Embedding sample: [-0.006819797679781914, -0.012808801606297493, -0.0016777735436335206, 0.0084628984

### Upload + Test

In [34]:
PINECONE_API_KEY = "pcsk_4Zrgdk_JE48SUN5TDzkNnqYWdbszMCwwkJpQpQLq5MxQDw4a7vJGyiWEMeEJMWhv9CWADB"
PINECONE_ENV = "us-east-1"
INDEX_NAME = "internal-docs"



pc = Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)
index = pc.Index(INDEX_NAME)

index.describe_index_stats()  # Check index stats

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5,
 'vector_type': 'dense'}

In [ ]:
# === Upload in batches ===
batch_size = 100
for start in tqdm(range(0, len(vector_data), batch_size), desc="Upserting to Pinecone"):
    batch = vector_data[start:start + batch_size]
    index.upsert(vectors=batch)

print(f"\nUploaded {len(vector_data)} chunks from '{filename}' to Pinecone index '{INDEX_NAME}'")

Upserting to Pinecone:   0%|          | 0/1 [00:00<?, ?it/s]

Upserting to Pinecone: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]


✅ Uploaded 4 chunks from 'formatted_text.txt' to Pinecone index 'internal-docs'


In [39]:
response = index.fetch(ids=["formatted_text.txt_0"])
pprint(response)


FetchResponse(namespace='',
              vectors={'formatted_text.txt_0': Vector(id='formatted_text.txt_0',
                                                      values=[-0.0215701945,
                                                              -0.0273689218,
                                                              -0.011659665,
                                                              0.0416276902,
                                                              0.00268283533,
                                                              0.00389349461,
                                                              0.0091900276,
                                                              0.0257571395,
                                                              0.0140300607,
                                                              0.0362079553,
                                                              0.0256458838,
                                                   

In [40]:
query = "What should I do to maintain a clean and organized workspace?"
query_vector = model.encode([query])[0].tolist()

result = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

for match in result['matches']:
    print(f"\n--- ID: {match['id']} | Score: {match['score']} ---")
    print(match['metadata']['text'])



--- ID: formatted_text.txt_1 | Score: 0.849586666 ---
2. Organizing Work Materials:
o Sort through physical documents. Shred or dispose of sensitive documents no longer
needed. Remove all name cards and old files outside the bin near the lift area.
o Return all company property (e.g., laptop, keyboard, employment card, medical card, office
keys etc) to the designated person or department.
o Please ensure that sensitive documents are not left on display, which can cause data theft
and leaks.
3. Desk Equipment:
o Do a clean up and leave the desk and its equipment (e.g., monitors, keyboards etc) clean
and functional.
Digital File Handover
Employees must ensure that all relevant digital files are properly backed up and handed over. The following
steps should be completed:
1. Backup Necessary Files:
o Transfer all work-related files to the designated shared drive or cloud storage.
o Ensure files are organized logically in folders for easy access.
2. Handover of Credentials:

--- ID: format